In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
TRAIN_PATH = r"C:\Users\nihal\Desktop\NLP_Project\training_pairs.csv"

df = pd.read_csv(TRAIN_PATH)
print("Total training pairs:", len(df))
df.head()  

Total training pairs: 130790


,query,positive_passage,negative_passage
0,cortana what vitamin is niacin.,Vitamin B3 is one of 8 B vitamins. It is also ...,Spinach is also an excellent source of vitamin...
1,cortana what vitamin is niacin.,Vitamin B3 is one of 8 B vitamins. It is also ...,Definition. Vitamin A deficiency exists when t...
2,cortana what vitamin is niacin.,Vitamin B3 is one of 8 B vitamins. It is also ...,Celery is an excellent source of vitamin K and...
3,cortana what vitamin is niacin.,Vitamin B3 is one of 8 B vitamins. It is also ...,What is Vitamin A Good for. Vitamins are essen...
4,cortana what vitamin is niacin.,Vitamin B3 is one of 8 B vitamins. It is also ...,Vitamin H (Biotin) Biotin is a vitamin that is...


In [4]:
SANITY_SIZE = 25000
df_sanity = df.sample(n=SANITY_SIZE, random_state=42).reset_index(drop=True)

print("Sanity training size:", len(df_sanity))

Sanity training size: 25000


In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 128

In [6]:
class RankingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def encode(self, query, passage):
        return self.tokenizer(
            query,
            passage,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        pos_enc = self.encode(row["query"], row["positive_passage"])
        neg_enc = self.encode(row["query"], row["negative_passage"])

        return {
            "pos_input_ids": pos_enc["input_ids"].squeeze(0),
            "pos_attention_mask": pos_enc["attention_mask"].squeeze(0),
            "neg_input_ids": neg_enc["input_ids"].squeeze(0),
            "neg_attention_mask": neg_enc["attention_mask"].squeeze(0),
        }


In [7]:
sanity_dataset = RankingDataset(df_sanity, tokenizer, MAX_LEN)

sanity_loader = DataLoader(
    sanity_dataset,
    batch_size=4,   
    shuffle=True
)

len(sanity_loader)


6250

In [8]:
class CrossEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.linear = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0]
        score = self.linear(cls_output)
        return score.squeeze(-1)

In [ ]:
model = CrossEncoder().to(device)

criterion = nn.MarginRankingLoss(margin=1.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [10]:
EPOCHS = 10

model.train()
total_loss = 0.0

for batch in tqdm(sanity_loader, desc="Sanity training"):

    pos_ids = batch["pos_input_ids"].to(device)
    pos_mask = batch["pos_attention_mask"].to(device)

    neg_ids = batch["neg_input_ids"].to(device)
    neg_mask = batch["neg_attention_mask"].to(device)

    pos_scores = model(pos_ids, pos_mask)
    neg_scores = model(neg_ids, neg_mask)

    target = torch.ones_like(pos_scores)

    loss = criterion(pos_scores, neg_scores, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

print("Sanity training average loss:", total_loss / len(sanity_loader))

Sanity training: 100%|██████████| 6250/6250 [16:22<00:00,  6.36it/s]

Sanity training average loss: 0.14628430530786515


In [11]:
MODEL_OUT = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project\cross_encoder_cpu.pt"
torch.save(model.state_dict(), MODEL_OUT)

print("Model saved to:", MODEL_OUT)

Model saved to: C:\Users\nihal\OneDrive\Desktop\NLP_Project\cross_encoder_cpu.pt
